# 08 - Cluster profiling

Objetivo: asignar clientes a una configuracion reproducible y caracterizar cada grupo con variables interpretables. Los nombres se asignan despues de observar los resultados.

In [ ]:
import sys
from pathlib import Path
import polars as pl
import plotly.express as px
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, build_customer_features, transform_features, run_clustering, profile_clusters, save_table

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
features = build_customer_features(load_transactions(ROOT / "transacciones_sinteticas.csv"))
matrix, _ = transform_features(features, scaler="robust")
labels = run_clustering(matrix, algorithm="kmeans", n_clusters=4)
profiles = profile_clusters(features, labels)
assignments = features.select("codigo_cliente").with_columns(pl.Series("cluster", labels))
save_table(assignments, ROOT / "results" / "models" / "customer_cluster_assignments.csv")
save_table(profiles, ROOT / "results" / "metrics" / "cluster_profiles.csv")
profiles

In [ ]:
plot_data = profiles.select(["cluster", "customers", "transaction_count_mean", "gross_amount_mean", "type_hhi_mean", "type_entropy_mean"]).to_dicts()
fig = px.scatter(plot_data, x="transaction_count_mean", y="gross_amount_mean", size="customers", color="cluster", hover_data=["type_hhi_mean", "type_entropy_mean"], title="Perfiles de clusters")
fig.show()

## Conclusion y recomendaciones
Los perfiles muestran diferencias observadas, no categorias predefinidas. Antes de comunicar segmentos se debe revisar estabilidad, clientes anomalos y la sensibilidad a la configuracion elegida.